# 数据切分为什么会泄漏？

**面试回答主线：**泄漏是训练过程使用了预测时刻不可用的信息，或验证集不再独立。先定义预测时点，再 split，所有统计量与特征只能在训练窗口拟合。本实验模拟退款风险预测，展示未来“最终退款状态”作为特征时如何制造虚高分数。

## 真实案例

风控系统在支付完成时预测订单是否会退款。可用特征是金额和过去 7 天投诉次数；“最终是否退款”和“退款完成金额”在支付时尚不存在，放进训练列就是未来泄漏。最后四天订单被视为上线后的时间回放。

In [1]:
import numpy as np  # 导入 NumPy 手写预处理和逻辑回归。
np.set_printoptions(precision=3, suppress=True)  # 设置易读的数值显示格式。
order = np.array(['F01', 'F02', 'F03', 'F04', 'F05', 'F06', 'V01', 'V02', 'V03', 'V04'])  # 构造支付订单编号。
amount = np.array([39, 220, 55, 330, 70, 410, 48, 280, 95, 360], dtype=float)  # 记录支付时可见的订单金额。
complaints = np.array([0, 3, 0, 4, 1, 5, 0, 3, 1, 4], dtype=float)  # 记录支付时可见的历史投诉次数。
refund = np.array([0, 1, 0, 1, 0, 1, 1, 0, 1, 0], dtype=float)  # 记录订单结束后才得到的退款标签并构造时间外模式反转。
future_refund_amount = amount * refund  # 故意构造预测时不可用的退款完成金额字段。
train_index = np.arange(6)  # 将前六个日期设为训练窗口。
valid_index = np.arange(6, 10)  # 将后四个日期设为未来回放窗口。
print('订单 | 支付金额 | 历史投诉 | 最终退款 | 退款完成金额')  # 输出完整账本字段以定位泄漏列。
for index in range(len(order)):  # 逐条展示带事件时间语义的订单。
    print(f'{order[index]} | {amount[index]:8.0f} | {complaints[index]:8.0f} | {int(refund[index])} | {future_refund_amount[index]:12.0f}')  # 输出一条订单及其未来字段。

订单 | 支付金额 | 历史投诉 | 最终退款 | 退款完成金额
F01 |       39 |        0 | 0 |            0
F02 |      220 |        3 | 1 |          220
F03 |       55 |        0 | 0 |            0
F04 |      330 |        4 | 1 |          330
F05 |       70 |        1 | 0 |            0
F06 |      410 |        5 | 1 |          410
V01 |       48 |        0 | 1 |           48
V02 |      280 |        3 | 0 |            0
V03 |       95 |        1 | 1 |           95
V04 |      360 |        4 | 0 |            0


## Baseline / 基线

基线是业务规则：历史投诉至少 3 次则进入人工复核。它只依赖支付时可用的历史信息。

In [2]:
def binary_accuracy(probability, target):  # 定义将概率阈值化后计算准确率的函数。
    return float(np.mean((probability >= 0.5) == target))  # 返回分类正确比例。
baseline_probability = (complaints[valid_index] >= 3).astype(float)  # 用投诉阈值生成未来订单的基线预测。
baseline_accuracy = binary_accuracy(baseline_probability, refund[valid_index])  # 计算基线在未来回放上的准确率。
print('投诉规则预测:', baseline_probability.astype(int))  # 展示基线逐订单结果。
print(f'投诉规则未来准确率={baseline_accuracy:.3f}')  # 输出基线指标。

投诉规则预测: [0 1 0 1]
投诉规则未来准确率=0.000


In [3]:
def fit_logistic(train_feature, train_target, step_size=0.25, steps=300):  # 定义仅用 NumPy 的逻辑回归训练函数。
    mean = train_feature.mean(axis=0)  # 只根据训练窗口计算均值。
    std = train_feature.std(axis=0) + 1e-6  # 只根据训练窗口计算标准差并避免除零。
    x = (train_feature - mean) / std  # 标准化训练特征。
    x = np.c_[np.ones(len(x)), x]  # 添加截距列。
    weight = np.zeros(x.shape[1])  # 初始化线性权重。
    for step in range(steps):  # 迭代执行梯度下降。
        probability = 1.0 / (1.0 + np.exp(-(x @ weight)))  # 计算当前退款概率。
        gradient = x.T @ (probability - train_target) / len(x)  # 计算交叉熵的平均梯度。
        weight -= step_size * gradient  # 更新权重以降低训练损失。
    return mean, std, weight  # 返回部署时需要固化的预处理统计量和权重。
def predict_logistic(feature, mean, std, weight):  # 定义部署阶段的逻辑回归预测函数。
    x = (feature - mean) / std  # 复用训练统计量而不触碰未来数据。
    x = np.c_[np.ones(len(x)), x]  # 添加与训练一致的截距列。
    return 1.0 / (1.0 + np.exp(-(x @ weight)))  # 返回预测概率。
safe_feature = np.c_[amount, complaints]  # 构造支付时真正可用的两列特征。
safe_mean, safe_std, safe_weight = fit_logistic(safe_feature[train_index], refund[train_index])  # 在训练窗口拟合安全模型。
safe_probability = predict_logistic(safe_feature[valid_index], safe_mean, safe_std, safe_weight)  # 对未来窗口生成安全概率。
safe_accuracy = binary_accuracy(safe_probability, refund[valid_index])  # 计算安全模型真实可解释的未来表现。
print('安全模型权重:', np.round(safe_weight, 3))  # 展示只由可用特征学到的参数。
print('安全模型未来概率:', np.round(safe_probability, 3))  # 展示未来订单的风险概率。

安全模型权重: [0.668 2.703 2.951]
安全模型未来概率: [0.005 0.975 0.056 0.999]


In [4]:
leaky_feature = np.c_[amount, complaints, future_refund_amount]  # 故意加入支付后才产生的退款金额。
leaky_mean, leaky_std, leaky_weight = fit_logistic(leaky_feature[train_index], refund[train_index])  # 错误地在含未来字段的数据上训练模型。
leaky_probability = (future_refund_amount[valid_index] > 0.0).astype(float)  # 错误地直接把未来退款完成金额当作泄漏预测。
leaky_accuracy = binary_accuracy(leaky_probability, refund[valid_index])  # 计算被泄漏污染的虚高指标。
print('模型        | 未来准确率 | 可用特征')  # 输出比较表头。
print(f'投诉规则    | {baseline_accuracy:10.3f} | 历史投诉')  # 输出规则基线行。
print(f'安全逻辑回归| {safe_accuracy:10.3f} | 金额+历史投诉')  # 输出安全模型行。
print(f'泄漏逻辑回归| {leaky_accuracy:10.3f} | 加入退款完成金额')  # 输出错误模型行。

模型        | 未来准确率 | 可用特征
投诉规则    |      0.000 | 历史投诉
安全逻辑回归|      0.000 | 金额+历史投诉
泄漏逻辑回归|      1.000 | 加入退款完成金额


## 结果解读

泄漏模型利用退款完成金额这一未来字段，会显示比安全模型漂亮得多的离线结果；它在真实支付时没有这个输入，因而根本无法部署。指标越完美，越应该先检查字段可见时间和目标代理。

In [5]:
print('订单 | 真实退款 | 安全概率 | 泄漏概率')  # 输出逐订单诊断表头。
for local_index, global_index in enumerate(valid_index):  # 逐条对比安全和泄漏预测。
    print(f'{order[global_index]} | {int(refund[global_index])}        | {safe_probability[local_index]:8.3f} | {leaky_probability[local_index]:8.3f}')  # 输出同一订单的两种预测。
print('解释：安全模型的分数才可作为上线前回放指标，泄漏模型只能作为反例。')  # 明确正确的评估结论。

订单 | 真实退款 | 安全概率 | 泄漏概率
V01 | 1        |    0.005 |    1.000
V02 | 0        |    0.975 |    0.000
V03 | 1        |    0.056 |    1.000
V04 | 0        |    0.999 |    0.000
解释：安全模型的分数才可作为上线前回放指标，泄漏模型只能作为反例。


## 失败案例与修复

另一个常见错误是先在全量数据上做标准化，再切分。下面将安全流程和错误流程的均值并列：后者把未来高金额订单的统计量泄回训练阶段。修复是先 split，再在训练窗口 `fit`，在未来窗口只 `transform`。

In [6]:
global_mean = amount.mean()  # 错误地使用未来订单计算全量均值。
train_mean = amount[train_index].mean()  # 正确地只使用训练订单计算均值。
wrong_scaled_train = (amount[train_index] - global_mean) / amount.std()  # 构造泄漏统计量下的训练特征。
right_scaled_train = (amount[train_index] - train_mean) / amount[train_index].std()  # 构造训练窗口统计量下的训练特征。
print(f'失败：全量金额均值={global_mean:.1f}，训练时已经看到未来价格分布。')  # 输出泄漏统计量证据。
print(f'修复：训练金额均值={train_mean:.1f}，未来订单不参与拟合。')  # 输出安全统计量。
print('前两条错误缩放:', np.round(wrong_scaled_train[:2], 3))  # 展示错误预处理的中间结果。
print('前两条正确缩放:', np.round(right_scaled_train[:2], 3))  # 展示安全预处理的中间结果。
print('生产差距：需维护 point-in-time 特征表、数据血缘、切分键、快照版本与训练—服务一致性检查。')  # 描述生产中防泄漏的系统机制。

失败：全量金额均值=190.7，训练时已经看到未来价格分布。
修复：训练金额均值=187.3，未来订单不参与拟合。
前两条错误缩放: [-1.099  0.212]
前两条正确缩放: [-1.031  0.227]
生产差距：需维护 point-in-time 特征表、数据血缘、切分键、快照版本与训练—服务一致性检查。


In [7]:
assert len(order) >= 5  # 保护案例包含至少五条带业务语义的订单。
assert leaky_accuracy >= safe_accuracy  # 保护未来字段会虚高离线分数这一反例。
assert global_mean != train_mean  # 保护未来窗口确实改变了全量统计量。
assert safe_feature.shape[1] == 2  # 保护安全模型没有偷偷使用退款完成字段。